# Test exceptions notebook:

This notebook is used to test the implementation of **jklab-core/exceptions.py** module.

Testing (roughly) follows this routine:

0. import the module to test;
1. select component (variable/class/function) to test;
2. enstablish the correct behaviour;
3. implement a local function to assert the results;
4. summarise which tests have been passed.

## Module import

In [31]:
import pytest 

import jklab.core.exceptions as jkex

## Error helpers

In [32]:
# Collection of all test results in this notebook
test_results = {}

# Single test record function
def record_test(
    test_name,
    condition
):
    """
    Record and display the result of a test.
    """

    passed_flag = bool(condition)

    test_results[test_name] = passed_flag

    if passed_flag:
        print(f"✅ PASSED: {test_name}.")
    else:
        print(f"❌ FAILED: {test_name}.")


# Test summary function
def print_test_summary(
    test_results
):
    """
    Print a summary of test results.
    """

    # Used for visual separation
    separator_len = 40

    total = len(test_results)
    passed = sum(test_results.values())
    failed = total - passed

    print()
    print("=" * separator_len)
    print("TEST SUMMARY")
    print("=" * separator_len)

    print(f"Passed: {passed}/{total}")
    print(f"Failed: {failed}/{total}")

    if total:
        ratio = passed / total * 100
        print(f"Success rate: {ratio:.1f}%")

    print()

    for name, result in test_results.items():

        status = "PASSED" if result else "FAILED"

        print(f"{status}: {name}")

    print("=" * separator_len)

## Test 1 - Custom exception class

In [33]:
def test_jkl_error(
    error,
    message
):
    """
    Test the JKateLabError and derived classes.

    Inputs: a type of error and relative message.

    Output: True if the error message matches with
        the input message; False otherwise.
    """

    try:

        with pytest.raises(error) as exc_info:

            raise error(message)

        return str(exc_info.value) == message

    except Exception:

        return False

In [34]:
# === Input ===
# Pick an error and a message
error = jkex.JKateLabError
message = "Something went wrong."

# === Test & Record ===
record_test(
    test_name="JKateLabError: raise correct error",
        condition=test_jkl_error(
        error=error,
        message=message
    )
)

✅ PASSED: JKateLabError: raise correct error.


## Test 2 - Message formatting

In [35]:
def build_error_message(
    message,
    context=None
):
    """
    Build an error message.

    Inputs: custom message and context (optional).

    Outputs: formatted message.
    """

    lines = [message]
    
    if context:

        lines.append("")
        lines.append("Context:")

        for name, value in context.items():

            lines.append(
                f"{name} = {value!r}"
            )

    error_msg = "\n".join(lines)

    return error_msg


def test_format_error_message(
    message,
    context=None
):
    """
    Test format_error_message.

    Inputs: custom message and context (optional).

    Output: True if the message is correctly 
        formatted; False otherwise.
    """

    
    error_msg_jkl = jkex.format_message(
        message=message,
        context=context
    )

    error_msg_test = build_error_message(
        message=message,
        context=context
    )

    return error_msg_jkl == error_msg_test

### Test w/ no context

In [36]:
# === Input ===
# Pick a message
message = "Something went wrong."

# === Test w/o context ===
record_test(
    test_name=("format_error_message: "
               "error message formatting "
               "without context"),
    condition=test_format_error_message(
        message=message,
    )
)

✅ PASSED: format_error_message: error message formatting without context.


### Test w/ empty context

In [37]:
# === Input ===
# Pick a message and context (optional)
message = "Something went wrong."
context_dict = {}

# === Test w/ context ===
record_test(
    test_name=("format_error_message: "
               "error message formatting "
               "with empty context"),
    condition=test_format_error_message(
        message=message,
        context=context_dict
    )
)


✅ PASSED: format_error_message: error message formatting with empty context.


### Test w/ context

In [38]:
# === Input ===
# Pick a message and context (optional)
message = "Something went wrong."
context_dict = {
    "a": 1,
    "b": 2
}

# === Test w/ context ===
record_test(
    test_name=("format_error_message: "
               "error message formatting "
               "with context"),
    condition=test_format_error_message(
        message=message,
        context=context_dict
    )
)

✅ PASSED: format_error_message: error message formatting with context.


## Test 3 - Error raising

In [39]:
def test_raise_error(
    error,
    message,
    context=None
):
    """
    Test raise_error.

    Inputs: a type of error, custom message and
        context (optional)

    Output: True if raised correct error and 
        the error message is correctly formatted;
        False otherwise.
    """

    try:

        jkex.raise_error(
            error=error,
            message=message,
            context=context
        )

    except error as exc_info:

        expected_msg = jkex.format_message(
            message=message,
            context=context,
        )

        return str(exc_info) == expected_msg

    return False

### Custom JKateLab exception

In [40]:
# === Input ===
# Pick an error, a message and context (optional)
error = jkex.JKateLabError
message = "Something went wrong."
context_dict = {
    "a": 1,
    "b": 2
}

# === Test ===
record_test(
    test_name=("raise_error: "
               "raise custom error with context"),
    condition=test_raise_error(
        error=error,
        message=message,
        context=context_dict
    )
)

✅ PASSED: raise_error: raise custom error with context.


### Native Python exception

In [41]:
# === Input ===
# Pick an error, a message and context (optional)
error = ValueError
message = "Something went wrong."
context_dict = {
    "a": 1,
    "b": 2
}

# === Test ===
record_test(
    test_name=("raise_error: "
               "raise native Python exception with context"),
    condition=test_raise_error(
        error=error,
        message=message,
        context=context_dict
    )
)

✅ PASSED: raise_error: raise native Python exception with context.


## Test 5 - Warning 'raising'

In [42]:
def test_raise_warning(
    warning,
    message,
    context=None
):
    """
    Test raise_warning.

    Inputs: a type of warning, custom message and
        context (optional)

    Output: True if raised correct warning and 
        the warning message is correctly formatted;
        False otherwise.
    """

    try:

        with pytest.warns(warning) as warning_info:

            jkex.raise_warning(
                warning=warning,
                message=message,
                context=context,
            )

        expected_message = jkex.format_message(
            message=message,
            context=context,
        )

        return str(warning_info[0].message) == expected_message

    except Exception:

        return False

### Custom JKateLab warning

In [43]:
# === Input ===
# Pick an error, a message and context (optional)
warning = jkex.JKateLabWarning
message = "Something went wrong."
context_dict = {
    "a": 1,
    "b": 2
}

# === Test ===
record_test(
    test_name=("raise_warning: "
               "raise custom warning with context"),
    condition=test_raise_warning(
        warning=warning,
        message=message,
        context=context_dict
    )
)

✅ PASSED: raise_warning: raise custom warning with context.


### Native Python warning

In [44]:
# === Input ===
# Pick an error, a message and context (optional)
warning = SyntaxWarning
message = "Something went wrong."
context_dict = {
    "a": 1,
    "b": 2
}

# === Test ===
record_test(
    test_name=("raise_warning: "
               "raise native Python warning with context"),
    condition=test_raise_warning(
        warning=warning,
        message=message,
        context=context_dict
    )
)

✅ PASSED: raise_warning: raise native Python warning with context.


## Summary

In [45]:
print_test_summary(test_results)


TEST SUMMARY
Passed: 8/8
Failed: 0/8
Success rate: 100.0%

PASSED: JKateLabError: raise correct error
PASSED: format_error_message: error message formatting without context
PASSED: format_error_message: error message formatting with empty context
PASSED: format_error_message: error message formatting with context
PASSED: raise_error: raise custom error with context
PASSED: raise_error: raise native Python exception with context
PASSED: raise_warning: raise custom warning with context
PASSED: raise_warning: raise native Python warning with context
